In [4]:
import re
import pandas as pd

# =====================================================
# 1. PATHS
# =====================================================
SCOPUS_PATH = "scopus_export_Mar 31-2026_fc3be87f-2e9d-4ea0-8c22-2e3d855f3cec.txt"
IEEE_PATH   = "IEEE Xplore Citation Plain Text Download 2026.3.31.16.1.40.txt"
GS_PATH     = "google_scholar.txt"

# =====================================================
# 2. LOAD FILES
# =====================================================
def load_file(path):
    try:
        return open(path, encoding="utf-8", errors="ignore").read()
    except FileNotFoundError:
        print(f"[WARNING] File not found: {path}")
        return ""

scopus = load_file(SCOPUS_PATH)
ieee   = load_file(IEEE_PATH)
gs     = load_file(GS_PATH)

# =====================================================
# 3. GENERIC CLEANERS
# =====================================================
def clean_text(x):
    if not x:
        return ""
    return re.sub(r"\s+", " ", x).strip()

def extract_year(text):
    m = re.search(r"(19|20)\d{2}", text)
    return m.group(0) if m else ""

# =====================================================
# 4. SCOPUS PARSER (IMPROVED)
# =====================================================
def parse_scopus_records(text):
    recs = []
    blocks = re.split(r"\n\s*\n", text)

    for b in blocks:
        if "DOI:" not in b and "TITLE" not in b.upper():
            continue

        title = ""
        authors = ""
        year = ""
        doi = ""

        # TITLE
        m = re.search(r"Title:\s*(.*)", b, re.I)
        if m:
            title = clean_text(m.group(1))
        else:
            lines = b.split("\n")
            if lines:
                title = clean_text(lines[0])

        # AUTHORS
        m = re.search(r"Authors?:\s*(.*)", b, re.I)
        if m:
            authors = clean_text(m.group(1))

        # YEAR
        year = extract_year(b)

        # DOI
        m = re.search(r"DOI:\s*([^\s]+)", b, re.I)
        if m:
            doi = m.group(1)

        recs.append({
            "Database": "Scopus",
            "Title": title,
            "Authors": authors,
            "Year": year,
            "DOI": doi,
            "Source": "Scopus"
        })

    return recs

# =====================================================
# 5. IEEE PARSER (IMPROVED)
# =====================================================
def parse_ieee_records(text):
    recs = []
    blocks = re.split(r"\n\s*\n", text)

    for b in blocks:
        title = ""
        authors = ""
        doi = ""
        year = ""

        # TITLE
        m = re.search(r"\"([^\"]+)\"", b)
        if m:
            title = clean_text(m.group(1))

        # AUTHORS (approx)
        m = re.search(r"\n(.*?)\n", b)
        if m:
            authors = clean_text(m.group(1))

        # DOI
        m = re.search(r"doi:\s*([^\s]+)", b, re.I)
        if m:
            doi = m.group(1)

        year = extract_year(b)

        if title:
            recs.append({
                "Database": "IEEE",
                "Title": title,
                "Authors": authors,
                "Year": year,
                "DOI": doi,
                "Source": "IEEE"
            })

    return recs

# =====================================================
# 6. GOOGLE SCHOLAR PARSER (UPGRADED)
# =====================================================
def parse_gs_records(text):
    recs = []
    blocks = re.split(r"-{5,}", text)

    for b in blocks:
        if "Title:" not in b:
            continue

        title = clean_text(re.search(r"Title:\s*(.*)", b).group(1)) if re.search(r"Title:", b) else ""
        authors = clean_text(re.search(r"Authors:\s*(.*)", b).group(1)) if re.search(r"Authors:", b) else ""
        year = clean_text(re.search(r"Year:\s*(.*)", b).group(1)) if re.search(r"Year:", b) else ""
        source = clean_text(re.search(r"Source:\s*(.*)", b).group(1)) if re.search(r"Source:", b) else ""

        recs.append({
            "Database": "Google Scholar",
            "Title": title,
            "Authors": authors,
            "Year": year,
            "DOI": "",
            "Source": source if source else "Google Scholar"
        })

    return recs

# =====================================================
# 7. RUN PARSING
# =====================================================
scopus_recs = parse_scopus_records(scopus)
ieee_recs   = parse_ieee_records(ieee)
gs_recs     = parse_gs_records(gs)

print("======== COUNTS ========")
print("Scopus:", len(scopus_recs))
print("IEEE:", len(ieee_recs))
print("Google Scholar:", len(gs_recs))

# =====================================================
# 8. MERGE
# =====================================================
df = pd.DataFrame(scopus_recs + ieee_recs + gs_recs)

# =====================================================
# 9. CLEAN + DEDUP
# =====================================================
df["Title_clean"] = (
    df["Title"]
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# priorité au DOI
df = df.sort_values(by="DOI", ascending=False)

df = df.drop_duplicates(subset=["DOI"], keep="first")
df = df.drop_duplicates(subset=["Title_clean"], keep="first")

# =====================================================
# 10. METADATA COLUMNS
# =====================================================
df.insert(0, "Record ID", [f"R{str(i+1).zfill(4)}" for i in range(len(df))])

for col in ["Domain", "Task", "Modality", "Fusion_Type", "Application",
            "Decision", "Reason", "Score"]:
    df[col] = ""

# =====================================================
# 11. SAVE
# =====================================================
df.to_csv("articles_clean.csv", index=False)
df.to_excel("articles_clean.xlsx", index=False)

print("✅ Done. Files saved.")

======== COUNTS ========
Scopus: 381
IEEE: 25
Google Scholar: 30


ModuleNotFoundError: No module named 'openpyxl'